# Association Rules: Complete Notebook
> **Course:** Machine Learning / Business Intelligence
> **Goal:** Discover co-occurrence patterns in transactions using Apriori and FP-Growth

---

## What will you learn?
1. Explore and preprocess transactional data
2. Encode transactions as a binary matrix
3. Mine frequent itemsets with **Apriori** and **FP-Growth**
4. Compute and interpret **Support**, **Confidence** and **Lift**
5. Visualize rules as an association network
6. Turn rules into business decisions

---
### Business context
We are analysts at an online supermarket. We want to find out which products are bought together in order to improve:
- Store **layout** and on-screen recommendations
- Personalized **email marketing**
- **Bundles** and cross-discounts

## 1. Installation and Setup

In [ ]:
!pip install mlxtend networkx --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#18181b',
    'axes.facecolor':   '#27272a',
    'axes.edgecolor':   '#3f3f46',
    'text.color':       '#e4e4e7',
    'axes.labelcolor':  '#a1a1aa',
    'xtick.color':      '#71717a',
    'ytick.color':      '#71717a',
    'grid.color':       '#3f3f46',
    'grid.alpha':        0.4,
    'axes.titlesize':    13,
    'axes.titlepad':     10,
    'figure.titlesize':  14,
})

EMERALD  = '#10b981'
AMBER    = '#f59e0b'
BLUE     = '#3b82f6'
PURPLE   = '#8b5cf6'
RED      = '#ef4444'
ZINC400  = '#a1a1aa'

np.random.seed(42)
print('Libraries loaded successfully')

---
## 2. Dataset: Online Supermarket

We generate **500 transactions** with realistic embedded patterns.

> **Teaching trick:** we deliberately insert known patterns (e.g. {diapers -> beer}) so that we can find them and validate the metrics by hand.

In [ ]:
# Base probability of each item (relative frequency in the supermarket)
ITEM_PROBS = {
    'leche':          0.62,
    'pan':            0.58,
    'huevos':         0.54,
    'mantequilla':    0.42,
    'queso':          0.38,
    'yogur':          0.33,
    'café':           0.44,
    'azúcar':         0.28,
    'arroz':          0.32,
    'pasta':          0.30,
    'pollo':          0.38,
    'carne':          0.28,
    'verduras':       0.48,
    'frutas':         0.52,
    'jugo':           0.32,
    'refresco':       0.27,
    'cerveza':        0.22,
    'vino':           0.18,
    'chips':          0.28,
    'chocolate':      0.32,
    'galletas':       0.24,
    'salsa_tomate':   0.25,
    'aceite':         0.30,
    'detergente':     0.22,
    'pañales':        0.12,
}

N = 500
transactions = []

# Base transactions (probabilistic)
for _ in range(N):
    basket = [item for item, p in ITEM_PROBS.items() if np.random.random() < p]
    if len(basket) >= 1:
        transactions.append(sorted(basket))

# Embedded patterns: artificially raise co-occurrence
patterns = [
    (['pañales', 'cerveza', 'chips'],          60),   # the classic
    (['leche', 'pan', 'mantequilla', 'huevos'], 55),  # breakfast
    (['pasta', 'salsa_tomate', 'queso'],        50),  # Italian combo
    (['pollo', 'arroz', 'verduras'],            45),  # healthy lunch
    (['café', 'azúcar', 'leche'],               48),  # coffee with milk
    (['vino', 'queso', 'galletas'],             30),  # appetizer
    (['chocolate', 'galletas', 'leche'],        35),  # afternoon snack
]
for items_list, reps in patterns:
    for _ in range(reps):
        extra = [i for i, p in ITEM_PROBS.items()
                 if np.random.random() < p * 0.4 and i not in items_list]
        transactions.append(sorted(set(items_list + extra)))

np.random.shuffle(transactions)

print(f'Total transactions:      {len(transactions)}')
print(f'Example transaction:     {transactions[0]}')
print(f'Largest basket:          {max(transactions, key=len)} ({max(len(t) for t in transactions)} items)')

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── General statistics ────────────────────────────────────────────
all_items  = [item for t in transactions for item in t]
basket_len = [len(t) for t in transactions]

item_counts = pd.Series(all_items).value_counts()
item_support = item_counts / len(transactions)

print('═' * 50)
print(f'  Total transactions     : {len(transactions)}')
print(f'  Unique items           : {item_counts.nunique()}')
print(f'  Avg items per basket   : {np.mean(basket_len):.1f}')
print(f'  Matrix density         : {sum(basket_len) / (len(transactions) * item_counts.nunique()):.3f}')
print('═' * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Exploration: Online Supermarket', fontsize=14, color='#e4e4e7')

# Item frequency
top_items = item_support.head(15)
colors_bar = [EMERALD if s > 0.4 else BLUE if s > 0.25 else ZINC400 for s in top_items]
axes[0].barh(top_items.index[::-1], top_items.values[::-1], color=colors_bar[::-1], edgecolor='none')
axes[0].set_xlabel('Support (fraction of transactions)')
axes[0].set_title('Top 15 Items by Support')
axes[0].axvline(0.4, color=AMBER, linewidth=1, linestyle='--', label='threshold 0.40')
axes[0].legend()
axes[0].grid(axis='x')

# Basket size distribution
bins = range(1, max(basket_len) + 2)
axes[1].hist(basket_len, bins=bins, color=PURPLE, alpha=0.8, edgecolor='#18181b', rwidth=0.85)
axes[1].axvline(np.mean(basket_len), color=AMBER, linewidth=2, linestyle='--',
                label=f'mean = {np.mean(basket_len):.1f}')
axes[1].set_xlabel('Number of items in the basket')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Basket Size Distribution')
axes[1].legend()
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

---
## 4. Preprocessing: Binary Encoding

Association rule algorithms require a **binary matrix** where:
- Each **row** = one transaction
- Each **column** = one possible item
- Each cell = `1` if the item is in the transaction, `0` otherwise

```
              leche  pan  huevos  mantequilla  queso  ...
Transaction 1   1     1     0         1          0
Transaction 2   1     0     1         0          1
Transaction 3   0     1     1         1          0
```

> **Key difference from Sequence Mining:** here **order does not matter**. {pan, leche} and {leche, pan} are the same transaction.

In [ ]:
# Encoding with mlxtend's TransactionEncoder
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_bin = pd.DataFrame(te_array, columns=te.columns_)

print(f'Binary matrix shape: {df_bin.shape}')
print(f'Rows = transactions, Columns = unique items')
print()
df_bin.head()

In [ ]:
# Visualize the binary matrix for the first 20 transactions
fig, ax = plt.subplots(figsize=(16, 5))
subset = df_bin.iloc[:20].astype(int)
sns.heatmap(subset, cmap=['#27272a', EMERALD], linewidths=0.3, linecolor='#18181b',
            cbar=False, ax=ax, yticklabels=range(1, 21))
ax.set_title('Binary Matrix: First 20 Transactions (green = present)', pad=12)
ax.set_xlabel('Items')
ax.set_ylabel('Transaction #')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. The Apriori Algorithm: Frequent Itemsets

### How does it work?
1. Find all **1-itemsets** with support >= `min_support`
2. Generate candidates of size `k+1` by combining the frequent itemsets of size `k`
3. Prune any candidate whose **subset is not frequent** (the anti-monotone principle)
4. Repeat until there are no candidates left

| min_support | Effect |
|---|---|
| High (0.30+) | Few itemsets, very common but obvious rules |
| Medium (0.10 to 0.20) | Balance between coverage and novelty |
| Low (<0.05) | Many itemsets, niche rules, higher computational cost |

In [ ]:
MIN_SUPPORT = 0.07   # <- try 0.05, 0.10, 0.15

frequent_apriori = apriori(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
frequent_apriori['length'] = frequent_apriori['itemsets'].apply(len)
frequent_apriori = frequent_apriori.sort_values('support', ascending=False)

print(f'Frequent itemsets found with support >= {MIN_SUPPORT}: {len(frequent_apriori)}')
print()
print('Distribution by size:')
print(frequent_apriori['length'].value_counts().sort_index().to_string())
print()
print('Top 10 frequent itemsets:')
frequent_apriori.head(10)[['itemsets','support','length']].to_string(index=False)

In [ ]:
# Effect of the support threshold on the number of frequent itemsets
thresholds = np.arange(0.03, 0.35, 0.02)
counts = [len(apriori(df_bin, min_support=t, use_colnames=True)) for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, counts, color=EMERALD, linewidth=2.5, marker='o', markersize=5)
ax.fill_between(thresholds, counts, alpha=0.15, color=EMERALD)
ax.axvline(MIN_SUPPORT, color=AMBER, linewidth=1.5, linestyle='--', label=f'current threshold = {MIN_SUPPORT}')
ax.set_xlabel('min_support')
ax.set_ylabel('Number of frequent itemsets')
ax.set_title('Sensitivity to the Support Threshold (Apriori)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

---
## 6. The FP-Growth Algorithm: Comparison with Apriori

**FP-Growth** builds a compressed prefix tree (the **FP-Tree**) and mines patterns without explicitly generating candidates.

| Dimension | Apriori | FP-Growth |
|---|---|---|
| Passes over the database | Multiple | 2 |
| Candidate generation | Yes | No |
| Memory usage | Low | Higher (tree) |
| Speed (large datasets) | Slow | Fast |
| Result | Identical | Identical |

In [ ]:
import time

t0 = time.perf_counter()
frequent_apriori2 = apriori(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
t_apriori = time.perf_counter() - t0

t0 = time.perf_counter()
frequent_fp = fpgrowth(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
t_fp = time.perf_counter() - t0

print(f'Apriori   -> {len(frequent_apriori2):4d} frequent itemsets in {t_apriori*1000:.1f} ms')
print(f'FP-Growth -> {len(frequent_fp):4d} frequent itemsets in {t_fp*1000:.1f} ms')
print(f'FP-Growth speedup: {t_apriori/t_fp:.1f}x')
print()
# Check that the results are identical
sets_a = set(frozenset(i) for i in frequent_apriori2['itemsets'])
sets_f = set(frozenset(i) for i in frequent_fp['itemsets'])
print(f'Identical results? {sets_a == sets_f}  <- must always be True')

---
## 7. Association Rules and Metrics

### Formal definition
A rule `X -> Y` means: *"when X occurs, Y tends to occur"*

| Metric | Formula | Interpretation |
|---|---|---|
| **Support** | `P(X ∪ Y)` | How common is the combination overall? |
| **Confidence** | `P(Y \| X) = P(X ∪ Y) / P(X)` | Given X, how sure can we be of Y? |
| **Lift** | `conf(X→Y) / P(Y)` | Is the rule better than chance? |
| **Leverage** | `P(X ∪ Y) - P(X)·P(Y)` | Excess co-occurrence relative to independence |
| **Conviction** | `(1 - P(Y)) / (1 - conf(X→Y))` | How strong the implication is (∞ = perfect) |

### Lift guide
- `lift = 1`: X and Y are **independent** (the rule adds no value)
- `lift > 1`: **positive association** (X favors Y)
- `lift < 1`: **negative association** (X suppresses Y)

In [ ]:
MIN_CONFIDENCE = 0.40
MIN_LIFT       = 1.20

rules = association_rules(frequent_fp, metric='lift', min_threshold=MIN_LIFT)
rules = rules[rules['confidence'] >= MIN_CONFIDENCE].copy()
rules = rules.sort_values('lift', ascending=False)

# Format for readability
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules['rule'] = rules['antecedents_str'] + '  →  ' + rules['consequents_str']

print(f'Rules found (conf >= {MIN_CONFIDENCE}, lift >= {MIN_LIFT}): {len(rules)}')
print()

display_cols = ['rule', 'support', 'confidence', 'lift', 'leverage']
rules[display_cols].head(15).style \
    .format({'support': '{:.3f}', 'confidence': '{:.3f}', 'lift': '{:.2f}', 'leverage': '{:.4f}'}) \
    .bar(subset=['lift'], color='#10b981', vmin=1) \
    .bar(subset=['confidence'], color='#3b82f6', vmin=0)

In [ ]:
# ── Manual computation to validate ────────────────────────────────
def calc_rule_manual(transactions, antecedent, consequent):
    """Compute support, confidence and lift by hand to check mlxtend."""
    n = len(transactions)
    ant_set = set(antecedent)
    con_set = set(consequent)
    both = sum(1 for t in transactions if ant_set <= set(t) and con_set <= set(t))
    ant_only = sum(1 for t in transactions if ant_set <= set(t))
    con_only = sum(1 for t in transactions if con_set <= set(t))
    support    = both / n
    confidence = both / ant_only if ant_only > 0 else 0
    lift       = confidence / (con_only / n) if con_only > 0 else 0
    return support, confidence, lift

# Check the rule: {pañales} -> {cerveza}
sup, conf, lift = calc_rule_manual(transactions, ['pañales'], ['cerveza'])
print('Rule: {pañales} -> {cerveza}')
print(f'  Support    = {sup:.4f}  ({sup*100:.1f}% of transactions contain both)')
print(f'  Confidence = {conf:.4f}  ({conf*100:.1f}% of diaper buyers also buy beer)')
print(f'  Lift       = {lift:.4f}  ({lift:.1f}x more likely than by chance)')
print()
# Check the rule: {pan, mantequilla} -> {leche}
sup2, conf2, lift2 = calc_rule_manual(transactions, ['pan', 'mantequilla'], ['leche'])
print('Rule: {pan, mantequilla} -> {leche}')
print(f'  Support    = {sup2:.4f}')
print(f'  Confidence = {conf2:.4f}')
print(f'  Lift       = {lift2:.4f}')

In [ ]:
# ── Metric distributions ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribution of Metrics across the Rules Found')

for ax, col, color, title in zip(
    axes,
    ['support', 'confidence', 'lift'],
    [EMERALD, BLUE, PURPLE],
    ['Support', 'Confidence', 'Lift']
):
    ax.hist(rules[col], bins=20, color=color, alpha=0.85, edgecolor='#18181b')
    ax.axvline(rules[col].mean(), color=AMBER, linewidth=1.5, linestyle='--',
               label=f'mean={rules[col].mean():.2f}')
    if col == 'lift':
        ax.axvline(1.0, color=RED, linewidth=1, linestyle=':', label='lift=1 (chance)')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.legend(fontsize=8)
    ax.grid(axis='y')

plt.tight_layout()
plt.show()

---
## 8. Visualizations

In [ ]:
# ── Scatter: Support vs Confidence (size = lift) ──────────────────
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    s=rules['lift'] * 30,
    cmap='plasma',
    alpha=0.75,
    edgecolors='none'
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Lift', color='#e4e4e7')
cbar.ax.yaxis.set_tick_params(color='#71717a')

# Annotate the top 5 rules by lift
top5 = rules.nlargest(5, 'lift')
for _, row in top5.iterrows():
    ax.annotate(
        f"{row['antecedents_str']} → {row['consequents_str']}",
        (row['support'], row['confidence']),
        textcoords='offset points', xytext=(8, 4),
        fontsize=7, color='#fbbf24',
        bbox=dict(boxstyle='round,pad=0.2', fc='#18181b', alpha=0.7)
    )

ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Support vs Confidence: color and size = Lift')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Lift heatmap among the top items ─────────────────────────────
top_items_lift = list(item_support.head(12).index)
lift_matrix = pd.DataFrame(index=top_items_lift, columns=top_items_lift, dtype=float)

for a in top_items_lift:
    for b in top_items_lift:
        if a == b:
            lift_matrix.loc[a, b] = np.nan
            continue
        row = rules[(rules['antecedents_str'] == a) & (rules['consequents_str'] == b)]
        lift_matrix.loc[a, b] = row['lift'].values[0] if len(row) > 0 else 1.0

fig, ax = plt.subplots(figsize=(10, 8))
mask = lift_matrix.isna()
sns.heatmap(
    lift_matrix.astype(float), ax=ax,
    cmap='RdYlGn', center=1.0, vmin=0.5, vmax=2.5,
    linewidths=0.3, linecolor='#18181b',
    mask=mask, annot=True, fmt='.2f', annot_kws={'size': 8}
)
ax.set_title('Lift Heatmap among the 12 Most Frequent Items\n(green > 1 = positive association, red < 1 = negative)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Association network ───────────────────────────────────────────
top_rules = rules.nlargest(30, 'lift')

G = nx.DiGraph()
for _, row in top_rules.iterrows():
    for a in row['antecedents']:
        for c in row['consequents']:
            G.add_edge(a, c, weight=row['lift'], confidence=row['confidence'])

fig, ax = plt.subplots(figsize=(13, 9))
ax.set_facecolor('#18181b')

pos = nx.spring_layout(G, seed=42, k=2.2)

# Nodes: size = item support
node_sizes = [item_support.get(n, 0.1) * 3500 for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=BLUE, node_size=node_sizes, alpha=0.85, ax=ax)

# Edges: width = lift
edges = G.edges(data=True)
edge_widths = [d['weight'] * 0.8 for _, _, d in edges]
edge_alphas = [min(d['confidence'], 1.0) for _, _, d in edges]
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.6,
                       edge_color=EMERALD, arrows=True,
                       arrowstyle='->', arrowsize=12, ax=ax)

nx.draw_networkx_labels(G, pos, font_size=9, font_color='#f4f4f5', ax=ax)

ax.set_title('Association Rule Network\n(edge width = Lift, node size = item Support)', pad=14)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 9. Business Analysis

Metrics are just numbers. The real value lies in **turning them into actions**.

In [ ]:
# ── Top rules by lift ─────────────────────────────────────────────
print('TOP 10 RULES BY LIFT (most surprising associations)')
print('═' * 80)
for _, row in rules.nlargest(10, 'lift').iterrows():
    print(f"  {row['antecedents_str']:25s} →  {row['consequents_str']:20s}"
          f"  lift={row['lift']:.2f}  conf={row['confidence']:.0%}  sup={row['support']:.3f}")

In [ ]:
# ── Useful rules for a specific item ─────────────────────────────
TARGET_ITEM = 'leche'   # <- change this item

print(f'RULES WHERE "{TARGET_ITEM}" IS THE ANTECEDENT (if they buy milk -> they also buy...)')
print('═' * 75)
r_ant = rules[rules['antecedents_str'].str.contains(TARGET_ITEM, na=False)] \
              .sort_values('lift', ascending=False)
for _, row in r_ant.head(8).iterrows():
    print(f"  → {row['consequents_str']:25s}  lift={row['lift']:.2f}  conf={row['confidence']:.0%}")

print()
print(f'RULES WHERE "{TARGET_ITEM}" IS THE CONSEQUENT (buyers of X also buy milk)')
print('═' * 75)
r_con = rules[rules['consequents_str'].str.contains(TARGET_ITEM, na=False)] \
              .sort_values('lift', ascending=False)
for _, row in r_con.head(8).iterrows():
    print(f"  {row['antecedents_str']:25s} →  leche  lift={row['lift']:.2f}  conf={row['confidence']:.0%}")

In [ ]:
# ── Rule asymmetry ────────────────────────────────────────────────
# Do A->B and B->A have the same confidence? Not necessarily
print('ASYMMETRY: Comparing A->B vs B->A')
print('═' * 70)
symmetric_pairs = []
for _, r1 in rules.iterrows():
    # Look for the reverse rule
    inv = rules[
        (rules['antecedents_str'] == r1['consequents_str']) &
        (rules['consequents_str'] == r1['antecedents_str'])
    ]
    if len(inv) > 0 and r1['antecedents_str'] < r1['consequents_str']:
        r2 = inv.iloc[0]
        symmetric_pairs.append({
            'A → B': f"{r1['antecedents_str']} → {r1['consequents_str']}",
            'conf(A→B)': f"{r1['confidence']:.0%}",
            'conf(B→A)': f"{r2['confidence']:.0%}",
            'lift':       f"{r1['lift']:.2f}",
        })

if symmetric_pairs:
    df_sym = pd.DataFrame(symmetric_pairs)
    print(df_sym.to_string(index=False))
else:
    print('  No symmetric pairs were found with the current thresholds.')

In [ ]:
# ── Executive summary of recommendations ─────────────────────────
print('EXECUTIVE SUMMARY: BUSINESS RECOMMENDATIONS')
print('═' * 65)

top3_lift  = rules.nlargest(3, 'lift')
top3_conf  = rules.nlargest(3, 'confidence')
top3_sup   = rules.nlargest(3, 'support')

print('\nFor SURPRISING RECOMMENDATIONS (highest lift):')
for _, r in top3_lift.iterrows():
    print(f"   {r['antecedents_str']} → {r['consequents_str']}  (lift={r['lift']:.2f})")

print('\nFor HIGH-CONVERSION CROSS-SELL (highest confidence):')
for _, r in top3_conf.iterrows():
    print(f"   {r['antecedents_str']} → {r['consequents_str']}  (conf={r['confidence']:.0%})")

print('\nFor MASS PROMOTIONS (highest support):')
for _, r in top3_sup.iterrows():
    print(f"   {r['antecedents_str']} → {r['consequents_str']}  (sup={r['support']:.3f})")

---
## 10. Exercises

### Exercise 1: Threshold sensitivity
Change `MIN_SUPPORT` to 0.03, 0.08 and 0.15. For each value, write down:
- How many rules are generated?
- What is their average lift?
- Do spurious rules appear (lift close to 1)?

### Exercise 2: Targeted rules
Change `TARGET_ITEM` to `'cerveza'` and `'chocolate'`. What differences do you find in the purchase patterns?

### Exercise 3: Confidence vs Lift
Find a rule where `confidence` is high (>0.6) but `lift` is close to 1. Why is that rule not useful even though it has high confidence?

### Exercise 4: Asymmetry
For the rule `{pañales} -> {cerveza}`, what is the confidence of the reverse rule `{cerveza} -> {pañales}`? Which direction is more useful for marketing?

### Exercise 5: Your own dataset
Modify the `ITEM_PROBS` dictionary to simulate an **electronics e-commerce store** (laptops, mice, keyboards, headphones, etc.) and rerun the full analysis.